In [ ]:
import darts.utils.timeseries_generation
import numpy as np
import pandas as pd
import plotly.express as px
import torch
from darts.metrics import err
from darts.models import GlobalNaiveSeasonal

from aare.evaluation.custom_metrics.daily_peak import dpd
from aare.evaluation.evaluation import historical_forecasts
from aare.feature_set import FeatureSet
from aare.features.registry import FEATURES
from aare.params import read_params

# Evaluation metrics for detailed analysis

Prototyping extraction of raw errors from a set of historical forecasts to do more detailed analysis on them, e.g. by season.

Some useful notes and charts below!

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
params = read_params()
tz = params["general"]["timezone"]

In [ ]:
data_transformers = None
model = GlobalNaiveSeasonal(input_chunk_length=24, output_chunk_length=1)
meta = {"features": {"targets": ["temp_bern"]}}

# using the real LR model with most thorough eval (stride 1, horizon 96),
# takes 15min just to simulate historical forecasts (so much worse than the 2 min DPD calc)
# meta, model, data_transformers = load_model(name="LR", version="dev")

In [ ]:
ds = FeatureSet(
    targets=FEATURES.get_many(meta["features"]["targets"]),
    future=FEATURES.get_many(meta["features"].get("future")),
    split_params=params["split"],
)
val_target, _, val_fc = ds.get_val()
len(val_target[0])

In [ ]:
stride = 1
horizon = 96
hf = historical_forecasts(model, val_target, horizon, stride, future_cov=val_fc, data_transformers=data_transformers)
len(hf[0])

In [ ]:
bt = model.backtest(
    val_target,
    historical_forecasts=hf,
    metric=[err, dpd],
    metric_kwargs=[{}, dict(tz=tz)],
    reduction=None,
    future_covariates=val_fc,
    data_transformers=data_transformers,
)
bt[0].shape

In [ ]:
res_err = model.residuals(
    val_target,
    historical_forecasts=hf,
    metric=err,
    last_points_only=False,
    future_covariates=val_fc,
    data_transformers=data_transformers,
)
len(res_err)

In [ ]:
# res_dpd = model.residuals(val, historical_forecasts=hf, metric=dpd, last_points_only=False, metric_kwargs=dict(tz=tz))

In [ ]:
np.concatenate(bt, axis=0).shape

In [ ]:
x = np.concatenate(bt, axis=0)
x = x.reshape(-1, x.shape[-1])
x.shape

In [ ]:
times = np.stack([fc.time_index.values for fc_l in hf for fc in fc_l])
times.shape

In [ ]:
run_ts = times[:, 0] - pd.Timedelta(1, "s")
run_ts.shape

In [ ]:
run_ts.repeat(times.shape[1]).reshape(-1, times.shape[1])

In [ ]:
np.expand_dims(np.stack([fc.time_index.values for fc_l in hf for fc in fc_l]), 2).shape

In [ ]:
df = pd.DataFrame(
    x, index=np.stack([fc.time_index.values for fc_l in hf for fc in fc_l]).ravel(), columns=["err", "dpd"]
)
df

In [ ]:
df["run_ts"] = run_ts.repeat(times.shape[1])
df

In [ ]:
df = df.reset_index(names="time")[["run_ts", "time", "err", "dpd"]]
# df.to_csv("test.csv", index=False)
df

In [ ]:
df["age"] = df["time"] - df["run_ts"]
df

In [ ]:
df["ae"] = df["err"].abs()
df["adpd"] = df["dpd"].abs()

In [ ]:
df.drop(["run_ts", "time"], axis="columns").groupby("age").mean()

In [ ]:
df.melt("age", ["err", "dpd", "ae", "adpd"], var_name="metric")

In [ ]:
# px.violin(df.melt("age", ["err", "dpd", "ae", "adpd"], var_name="metric"), x="age", y="value", facet_row="metric", box=True)

## Notes/TODOs

- For anything that does complex data manipulation and analytics, and isn't tied to darts, use polars, it might be worth it. But tbf you need proficiency in both anyway.
- The code that generates the report for the model should take as input something that can be constructed easily from the timescaledb database. E.g. a polars lazyframe with run_ts, time, err and dpd.
- If you want to analyze the residuals with regard to actual values, e.g. does the model overestimate temperature when the sun isn't shining?, you'll need to join with other data based on time (asof). That should also fall under the same logic that works without darts and is a step after calculating the residuals (caveat: when evaluating the model you already have the validation data in memory, reusing that somehow (transforming darts to polars instead of influx to polars) is probably worth it. The join can be agnostic though.
- Sidestepping darts metrics further is probably not worth it because the integration with automatic logging during training of torch models is really nice. You could do both; keep darts metric during training but eval after training would be done by transforming historical forecasts into polars or whatever and doing the calculation there; would 100% be faster. Then you need 3 DPD impls though, darts metric, polars aggregation and SQL aggregation.
- You'll probably have to re-implement DPD in SQL for real-time reporting in Grafana, but I think that's fine (just watch out for timezone stuff).
- Using historical forecasts once and then backtest once for the aggregated metrics and once for the point metrics works well. Using the residual function does not work because that can only take one metric.
- DPD implementation is really slow, depending on how fast models are to train, evaluation could become the bottleneck (for LR it's definitely the case).

In [ ]:
import polars as pl
import polars.selectors as cs
from aare.evaluation.evaluation import _get_raw_metrics

raw_err = _get_raw_metrics(model, hf, val_target, tz)
raw_err = pl.from_dataframe(raw_err)
raw_err

In [ ]:
df = (
    raw_err.lazy()
    .sort("time")
    .group_by("time")
    # .group_by_dynamic("time", every="1d")
    .agg(
        cs.float().abs().median(),
        cs.float().abs().quantile(0.25).name.suffix("_q25"),
        cs.float().abs().quantile(0.75).name.suffix("_q75"),
    )
    .sort("time")
    .with_columns(cs.float().rolling_mean(24 * 30, center=True))
    .collect()
)
df

In [ ]:
px.line(df, "time", ["err", "dpd"])

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(
    [
        go.Scatter(
            name="err",
            x=df["time"],
            y=df["err"],
            mode="lines",
        ),
        go.Scatter(
            name="Q75%",
            x=df["time"],
            y=df["err_q75"],
            mode="lines",
            line=dict(width=0),
            showlegend=False,
        ),
        go.Scatter(
            name="Q25%",
            x=df["time"],
            y=df["err_q25"],
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            fill="tonexty",
        ),
    ]
)

fig.update_layout(
    yaxis=dict(
        title=dict(
            text="Absolute Forecast Error [°C]",
        )
    ),
    title=dict(
        text="Forecast error over validation period",
        subtitle=dict(
            text="Prediction interval shows quantiles for all different forecast runs that predicted that time (all ages)"
        ),
    ),
    hovermode="x",
)

fig

In [ ]:
# then you could add an hline with the average metric.
# then another chart where all years are overlapped and aggregated a well. Same for a day to see which day-times are easier to predict (potentially filter by season).
# maybe add a slider to select only one (or a range) forecast ages so you can see how the errors increase the longer ago they were predicted
# this analysis might be able to tell you how to construct a simpler (faster) metric that correlates with DPD for optimization purposes.
# it's already roughly correlated with MAE, but maybe RMSE or MASE with snaive might be better.
# but beware, MASE will punish the model if it gets "easy" predicitons wrong, but we don't want it to optimize the "easy" winter.
# maybe use marimo's self-contained wasm notebook in read-only mode for the published report

In [ ]:
x = np.concatenate(bt, axis=0)
np.mean(x, axis=1)

In [ ]:
mae_madpd = np.mean(np.abs(x), axis=1)
mae_madpd

In [ ]:
err = x[:, :, 0]
rmse = np.sqrt(np.mean(err**2, axis=1))
rmse

In [ ]:
manual_agg_bt = np.stack([mae_madpd[:, 0], rmse, mae_madpd[:, 1]], axis=-1)
manual_agg_bt

In [ ]:
from aare.evaluation.custom_metrics.daily_peak import madpd
from darts.metrics import mae, rmse

agg_bt = model.backtest(
    val_target,
    historical_forecasts=hf,
    metric=[mae, rmse, madpd],
    metric_kwargs=[{}, {}, dict(tz=tz)],
    reduction=None,
    future_covariates=val_fc,
    data_transformers=data_transformers,
)
agg_bt[0].shape

In [ ]:
agg_bt

In [ ]:
np.allclose(agg_bt, manual_agg_bt)
# oki doki, let's do it manually to avoid duplicate metric calculation :)

In [ ]:
df = raw_err.to_pandas()
df

## From historical forecasts to table to metrics

In [ ]:
times = np.stack([fc.time_index.values for fc_l in hf for fc in fc_l])
times

In [ ]:
run_ts = times[:, 0] - pd.Timedelta(1, "s")
run_ts = run_ts.repeat(times.shape[1])
run_ts

In [ ]:
pred = np.concat([fc.values() for fc_l in hf for fc in fc_l])
pred

In [ ]:
hf_df = pd.DataFrame(pred, index=times.ravel(), columns=["pred"])
hf_df = hf_df.reset_index(names="time")
hf_df.insert(0, "run_ts", run_ts)  # add run_ts as first col
hf_df

In [ ]:
hf_df["age"] = hf_df["time"] - hf_df["run_ts"]
hf_df["lag"] = hf_df["age"] // pd.Timedelta(1, "h") + 1
hf_df

In [ ]:
actual = pd.concat([v.to_dataframe() for v in val_target])
actual = actual.rename(columns={"temp_bern": "actual"})
actual

In [ ]:
hf_df = hf_df.join(actual, on="time")
hf_df

In [ ]:
hf_df["err"] = hf_df["actual"] - hf_df["pred"]
hf_df

In [ ]:
def _localize(df, col):
    df[col] = df[col].dt.tz_localize("UTC").dt.tz_convert(tz)


_localize(hf_df, "time")
_localize(hf_df, "run_ts")

hf_df

In [ ]:
# group by run and date of the predicted time
daily_max = hf_df[["actual", "pred"]].groupby([hf_df["run_ts"], hf_df["time"].dt.date]).transform("max")
hf_df["dpd"] = daily_max["actual"] - daily_max["pred"]
hf_df

In [ ]:
hf_df["dpd_darts"] = np.concatenate(bt, axis=0).reshape(-1, 2)[:, 1]
hf_df["dpd_darts"].values

In [ ]:
hf_df["dpd"].values

In [ ]:
np.allclose(hf_df["dpd"].values, np.concatenate(bt, axis=0).reshape(-1, 2)[:, 1], atol=0.001)

In [ ]:
x = hf_df[["run_ts", "err", "dpd"]].copy()
x["err_sq"] = x["err"] ** 2
x[["err", "dpd"]] = x[["err", "dpd"]].abs()
df_agg = x.groupby("run_ts").agg("mean").rename(columns=dict(err="MAE", err_sq="RMSE", dpd="MADPD"))
df_agg["RMSE"] = np.sqrt(df_agg["RMSE"])
df_agg
# TODO rmse and compare with other

In [ ]:
manual_agg_bt

In [ ]:
df_agg.values[:, [0, 2, 1]]

In [ ]:
# yay again, no need for using slow backtest function from darts
np.allclose(df_agg.values[:, [0, 2, 1]], manual_agg_bt)

## Testing setup

In [ ]:
darts.utils.timeseries_generation.constant_timeseries(length=10, freq="h")

In [ ]:
darts.utils.timeseries_generation.linear_timeseries(0, 9, length=10, freq="h")